# SIC 2026 · Inteligencia Artificial
## Clase 13: Representación de texto, temas, embeddings y LSTM

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Terminamos el Capítulo 7, desde la lámina 114, con las mismas reseñas de productos Samsung de las clases anteriores.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 08:00 a 08:15 | 0. Retomar | | Cargar las reseñas y las funciones de las clases anteriores |
| 08:15 a 09:00 | 1. TF-IDF y similitud coseno | 114 a 125 | TF-IDF paso a paso y un buscador de reseñas parecidas |
| 09:00 a 09:30 | 2. Clasificación | 127 a 133 | Naive Bayes contra regresión logística |
| 09:30 a 09:40 | Pausa | | |
| 09:40 a 10:40 | 3. Modelado de temas | 135 a 154 | LSA y LDA paso a paso, y con las reseñas |
| 10:40 a 11:10 | 4. Embeddings y skip-gram | 156 a 167 | Vectores de palabras |
| 11:10 a 11:55 | 5. LSTM | 168 a 175 | Clasificar reseñas leyendo en orden |
| 11:55 a 12:00 | Cierre | | |

Cada paso tiene **qué vamos a hacer**, **el código** y **qué deberías ver**. Los ejercicios tienen verificación y las soluciones están al final.

---
# Parte 0. Retomar lo de la clase anterior

**Qué vamos a hacer:** instalar `gensim` (para la Parte 4), cargar las reseñas desde la API de Samsung (o el respaldo ficticio si la API no responde) y definir otra vez las funciones de las clases anteriores: `limpiar`, `preprocesar`, `tokens_simples` y `rellenar`.

In [ ]:
!pip install -q gensim

import re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

STOPWORDS = set(stopwords.words("spanish"))
STOPWORDS_OPINION = STOPWORDS - {"no", "sin", "nada", "nunca", "muy", "más", "poco"}
stemmer = SnowballStemmer("spanish")

def limpiar(texto):
    texto = re.sub(r"https?://\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"([!?.])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def preprocesar(texto, stopwords_usar=STOPWORDS):
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower())
    tokens = [t for t in texto.split() if t not in stopwords_usar and (len(t) > 2 or t == "no")]
    return " ".join(stemmer.stem(t) for t in tokens)

def tokens_simples(texto):
    """Tokens en minúsculas, solo letras, sin quitar stopwords (para skip-gram)."""
    return re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower()).split()

def rellenar(secuencias, largo):
    """Relleno con ceros al inicio hasta un largo fijo (Clase 12)."""
    matriz = np.zeros((len(secuencias), largo), dtype=int)
    for i, s in enumerate(secuencias):
        s = s[-largo:]
        matriz[i, -len(s):] = s
    return matriz

print("Listo.")

In [ ]:
URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {"Accept": "application/json", "X-Bazaarvoice-Api-Version": "5.5", "X-Bazaarvoice-Platform-Version": "2",
               "Bv-Bfd-Token": "17499,main_site,es_CL", "Origin": "https://www.samsung.com"}
SKUS = ["SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA"]

def descargar_resenas(skus, max_paginas=2):
    todas = []
    for sku in skus:
        for pagina in range(max_paginas):
            parametros = {"resource": "reviews", "action": "REVIEWS_N_STATS",
                          "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
                          "apiversion": "5.5", "displaycode": "17499-es_cl", "sort": "submissiontime:desc",
                          "limit": 100, "offset": pagina * 100}
            res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
            res.raise_for_status()
            resultados = res.json().get("response", {}).get("Results", [])
            todas.extend(resultados); time.sleep(1)
            if len(resultados) < 100:
                break
    return pd.DataFrame(todas)

RESPALDO = [
    (5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'),
    (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'),
    (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'),
    (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'),
    (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'),
    (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'),
    (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'),
    (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'),
    (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'),
    (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'),
    (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'),
    (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'),
    (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'),
    (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'),
    (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'),
    (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'),
    (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'),
    (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'),
    (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'),
    (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'),
    (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'),
    (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'),
    (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'),
    (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'),
    (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'),
    (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'),
    (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'),
    (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'),
    (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'),
    (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'),
    (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'),
    (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'),
    (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'),
    (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'),
    (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'),
    (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'),
    (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'),
    (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'),
    (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'),
    (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'),
    (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'),
    (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'),
    (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'),
    (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'),
    (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'),
    (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'),
    (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'),
    (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'),
    (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'),
    (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'),
    (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'),
    (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'),
    (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'),
    (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'),
    (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'),
    (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'),
    (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'),
    (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'),
    (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'),
    (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.'),
]

try:
    resenas = (descargar_resenas(SKUS).drop_duplicates(subset="Id")
               .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})[["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError("Muy pocas reseñas")
    FUENTE = "API de Samsung (Bazaarvoice)"
except Exception:
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"

resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["completo"] = (resenas["titulo"].fillna("") + ". " + resenas["texto"]).apply(limpiar)
resenas["para_temas"] = resenas["completo"].apply(preprocesar)
resenas["para_opinion"] = resenas["completo"].apply(lambda t: preprocesar(t, STOPWORDS_OPINION))
print(f"Fuente: {FUENTE} | Reseñas: {len(resenas)}")
resenas[["nota", "completo"]].head(3)

**Qué deberías ver:** la fuente (API o respaldo), el número de reseñas y las tres primeras.

---
# Parte 1. TF-IDF y similitud coseno (láminas 114 a 125)

En la Clase 11 vimos TF-IDF. Hoy agregamos la herramienta para **comparar** documentos.

## Paso 1.0. TF-IDF paso a paso (láminas 114 a 123)

Antes de usar `TfidfVectorizer` con las reseñas, hacemos el cálculo completo **a mano y con código** sobre 4 reseñas cortas, ya preprocesadas.

| Doc | Texto |
|---|---|
| 1 | batería carga batería |
| 2 | batería dura carga |
| 3 | pantalla brillo pantalla |
| 4 | pantalla brillo color |

### Paso A. Matriz documento-término (bolsa de palabras)

Una fila por documento, una columna por palabra, y en cada celda **cuántas veces aparece**. El orden de las palabras se pierde: por eso se llama "bolsa". Esta tabla es la **DTM** (lámina 116); si se transpone (palabras en filas) es la **TDM**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

docs_juguete = ["batería carga batería", "batería dura carga", "pantalla brillo pantalla", "pantalla brillo color"]
nombres_docs = ["doc 1", "doc 2", "doc 3", "doc 4"]

conteo_j = CountVectorizer()
TF = pd.DataFrame(conteo_j.fit_transform(docs_juguete).toarray(), columns=conteo_j.get_feature_names_out(), index=nombres_docs)
TF

**Qué deberías ver:** por ejemplo, el doc 1 tiene `batería = 2` y `carga = 1`.

### Paso B. TF: frecuencia del término

**TF = cuántas veces aparece la palabra en ese documento.** Es la misma tabla. Responde: ¿qué tan importante es esta palabra **dentro** de este documento? Su problema: una palabra que aparece en todos lados pesa igual que una característica.

### Paso C. DF: frecuencia de documentos

**DF = en cuántos documentos aparece la palabra** (no cuántas veces). Se calcula contando, por columna, cuántas celdas son mayores que 0.

In [ ]:
DF = (TF > 0).sum()
DF

**Qué deberías ver:** `batería`, `brillo`, `carga` y `pantalla` aparecen en 2 documentos; `dura` y `color`, en 1.

### Paso D. IDF: frecuencia inversa de documentos

Queremos lo **contrario** del DF: más peso a lo que aparece en pocos documentos. scikit-learn usa, con **n = 4**:

$$\\text{IDF} = \\ln\\left(\\frac{1 + n}{1 + DF}\\right) + 1$$

- DF = 2: ln(5/3) + 1 = **1,511**
- DF = 1: ln(5/2) + 1 = **1,916**

Por qué así: el **logaritmo** suaviza (una palabra 100 veces más rara no debe pesar 100 veces más); **sumar 1 arriba y abajo** evita dividir por cero; **sumar 1 al final** evita que una palabra presente en todos los documentos quede con peso 0. Los libros usan ln(n / DF): da otros números, pero la idea es la misma.

In [ ]:
n = len(docs_juguete)
IDF = np.log((1 + n) / (1 + DF)) + 1
IDF.round(3)

### Paso E. TF × IDF

Se multiplica cada conteo por el IDF de su palabra. Doc 1: batería = 2 × 1,511 = **3,022**; carga = 1 × 1,511 = **1,511**.

In [ ]:
TFIDF_sin_norma = TF * IDF          # pandas multiplica cada columna por el IDF de esa palabra
TFIDF_sin_norma.round(3)

### Paso F. Normalizar (norma L2)

Se divide cada fila por su **largo**: la raíz de la suma de sus valores al cuadrado.

Doc 1: largo = √(3,022² + 1,511²) = √11,415 = **3,379**. Entonces batería = 3,022 / 3,379 = **0,894** y carga = 1,511 / 3,379 = **0,447**.

**Para qué:** sin normalizar, un documento largo tendría números grandes solo por tener más palabras. Después, todas las filas tienen largo 1 y lo que importa es la **proporción** entre palabras.

In [ ]:
largos = np.sqrt((TFIDF_sin_norma ** 2).sum(axis=1))
TFIDF_final = TFIDF_sin_norma.div(largos, axis=0)
display(TFIDF_final.round(3))

# Comprobación: scikit-learn da exactamente lo mismo
sk = TfidfVectorizer().fit_transform(docs_juguete).toarray()
print("¿Igual que TfidfVectorizer?", np.allclose(TFIDF_final.values, sk))

**Qué deberías ver:** la tabla final y `True`. Fíjense en el doc 2: "dura" (0,668) pesa más que "batería" (0,526), aunque ambas aparecen una vez, porque "dura" es más rara en el corpus.

### Ejercicio 1
Calcula **a mano** el IDF de "color" (aparece en 1 de los 4 documentos) con `np.log` y guárdalo en `idf_color`.

In [ ]:
idf_color = None  # TU CÓDIGO AQUÍ
print(idf_color)

In [ ]:
revisar(lambda: np.isclose(idf_color, IDF["color"]))

## Paso 1.1. La similitud coseno

> **Definición. Similitud coseno:** mide qué tan parecida es la **dirección** de dos vectores. Vale 1 si apuntan en la misma dirección, 0 si no comparten nada.
>
> $$\cos(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$
>
> Arriba, el **producto punto**: se multiplican los valores posición por posición y se suman. Abajo, el **largo** de cada vector: la raíz de la suma de sus valores al cuadrado.

**Ejemplo a mano.** Tres documentos contados sobre el vocabulario [batería, pantalla, precio]:

- A = [2, 1, 0]
- B = [1, 1, 0]
- C = [0, 0, 3]

cos(A, B) = (2·1 + 1·1 + 0·0) / (√5 · √2) = 3 / 3,162 = **0,949**. Muy parecidos.

cos(A, C) = (0 + 0 + 0) / (√5 · 3) = **0**. No comparten ninguna palabra.

¿Por qué coseno y no distancia? Porque no le importa el **largo** del documento: una reseña corta y una larga que hablan de lo mismo quedan cerca.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

A, B, Cv = np.array([2, 1, 0]), np.array([1, 1, 0]), np.array([0, 0, 3])
a_mano = A @ B / (np.linalg.norm(A) * np.linalg.norm(B))     # @ es el producto punto; norm es el largo
print(f"A mano: {a_mano:.3f} | scikit-learn: {cosine_similarity([A], [B])[0, 0]:.3f} | cos(A, C): {cosine_similarity([A], [Cv])[0, 0]:.3f}")

**Con la matriz TF-IDF del Paso 1.0:** como cada fila ya tiene largo 1, el coseno es solo el producto punto. cos(doc 1, doc 2) = 0,894 × 0,526 + 0,447 × 0,526 = **0,705** (hablan de lo mismo); cos(doc 1, doc 3) = **0** (no comparten palabras).

In [ ]:
pd.DataFrame(cosine_similarity(TFIDF_final), index=nombres_docs, columns=nombres_docs).round(3)

### Ejercicio 2
Calcula **a mano** (con `@` y `np.linalg.norm`) la similitud coseno entre `B` y `D = [2, 2, 1]`. Guárdala en `cos_bd`.

In [ ]:
D = np.array([2, 2, 1])
cos_bd = None  # TU CÓDIGO AQUÍ
print(cos_bd)

In [ ]:
revisar(lambda: np.isclose(cos_bd, cosine_similarity([B], [D])[0, 0]))

## Paso 1.2. Un buscador de reseñas parecidas

**Qué vamos a hacer:** convertir las reseñas en TF-IDF y, dada una consulta escrita por nosotros, encontrar las 3 reseñas con mayor similitud coseno. Es la idea básica de cualquier buscador.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

vect = TfidfVectorizer()
X_tfidf = vect.fit_transform(resenas["para_temas"])

def buscar(consulta, n=3):
    v = vect.transform([preprocesar(limpiar(consulta))])          # la consulta pasa por el mismo camino que las reseñas
    similitudes = cosine_similarity(v, X_tfidf)[0]
    mejores = similitudes.argsort()[::-1][:n]
    for i in mejores:
        print(f"{similitudes[i]:.2f} | nota {resenas.loc[i, 'nota']} | {resenas.loc[i, 'completo'][:110]}")

buscar("la batería se descarga muy rápido")
print()
buscar("problemas para conectar con el teléfono")

**Qué deberías ver:** reseñas sobre la batería en la primera búsqueda y sobre la conexión en la segunda. En la segunda, "teléfono" no aparece en las reseñas (dicen "celular"), así que solo ayudan "problemas" y "conectar". Es el límite de TF-IDF: compara palabras, no significados. Lo retomamos en la Parte 4.

---
# Parte 2. Clasificación: Naive Bayes contra regresión logística (láminas 127 a 133)

## Paso 2.1. Naive Bayes con bolsa de palabras

Recordemos Naive Bayes del Capítulo 5. Para un texto, compara:

$$P(\text{positiva} \mid \text{palabras}) \propto P(\text{positiva}) \cdot \prod_{\text{palabra}} P(\text{palabra} \mid \text{positiva})$$

- $P(\text{positiva})$: qué proporción de reseñas son positivas.
- $P(\text{palabra} \mid \text{positiva})$: qué tan frecuente es esa palabra **en las reseñas positivas**.

Es "ingenuo" porque supone que las palabras aparecen de forma independiente. Aun así, para texto funciona sorprendentemente bien. Como multiplicar muchas probabilidades pequeñas da números diminutos, en la práctica se suman logaritmos (lámina 129).

`MultinomialNB` trabaja con **conteos**, así que usamos `CountVectorizer`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, f1_score

opiniones = resenas[resenas["nota"] != 3].copy()
opiniones["sentimiento"] = np.where(opiniones["nota"] >= 4, "positiva", "negativa")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Naive Bayes (conteos)": Pipeline([("bow", CountVectorizer()), ("modelo", MultinomialNB())]),
    "Logística (TF-IDF)": Pipeline([("tfidf", TfidfVectorizer()), ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000))]),
}
predicciones = {}
for nombre, modelo in modelos.items():
    predicciones[nombre] = cross_val_predict(modelo, opiniones["para_opinion"], opiniones["sentimiento"], cv=cv)
    print(f"===== {nombre}")
    print(classification_report(opiniones["sentimiento"], predicciones[nombre], digits=3))

**Qué deberías ver:** dos reportes con resultados parecidos. Comparen el **recall de la clase negativa** y la **precision**: cuando el modelo dice "negativa", ¿qué tan seguido acierta?, y de las negativas reales, ¿cuántas encuentra?

Un detalle importante: Naive Bayes no tiene `class_weight`. Como $P(\text{positiva})$ entra directo en la fórmula, con datos muy desbalanceados tiende a favorecer a la clase mayoritaria. Si trabajan con las reseñas reales, donde casi todas son positivas, es probable que lo noten.

### Ejercicio 3
Guarda en `mejor_f1_negativa` el **nombre** del modelo (una de las dos claves de `predicciones`) con mayor F1 en la clase negativa.

In [ ]:
mejor_f1_negativa = None  # TU CÓDIGO AQUÍ
print(mejor_f1_negativa)

In [ ]:
_f = {n: f1_score(opiniones["sentimiento"], p, pos_label="negativa") for n, p in predicciones.items()}
revisar(lambda: mejor_f1_negativa == max(_f, key=_f.get))

---
# Parte 3. Modelado de temas (láminas 135 a 154)

Tres formas de encontrar temas, de la más simple a la más elaborada:

| Método | Idea | Cada documento tiene |
|---|---|---|
| K-medias sobre TF-IDF | Agrupa documentos parecidos (Clase 11) | **Un** tema |
| LSA (análisis semántico latente) | SVD truncado de la matriz TF-IDF: cada componente es un tema | Una mezcla de temas (con pesos que pueden ser negativos) |
| LDA (asignación latente de Dirichlet) | Modelo probabilístico: cada documento es una mezcla de temas, y cada tema, una mezcla de palabras | Una **mezcla** de temas (probabilidades que suman 1) |

## Paso 3.1. LSA paso a paso (láminas 135 a 143)

**La idea.** En TF-IDF, "batería" y "carga" son columnas distintas aunque siempre aparezcan juntas. LSA (análisis semántico latente) busca **temas ocultos**: grupos de palabras que tienden a aparecer en los mismos documentos. "Latente" significa oculto: nadie dijo cuáles son los temas.

**La herramienta: SVD.** La descomposición en valores singulares escribe la matriz TF-IDF **M** (documentos × palabras) como el producto de tres matrices (láminas 137 a 140):

$$M = U \\cdot \\Sigma \\cdot V^T$$

| Matriz | Forma | Qué significa |
|---|---|---|
| **U** | documentos × temas | Cuánto pertenece cada documento a cada tema |
| **Σ** (sigma) | temas × temas, diagonal | Qué tan importante es cada tema, de mayor a menor |
| **Vᵀ** | temas × palabras | Cuánto pesa cada palabra en cada tema |

Una forma de decirlo: la matriz original responde "qué palabras usa cada documento". La SVD la separa en dos preguntas: qué temas tiene cada documento (U) y qué palabras forman cada tema (Vᵀ). Σ dice cuáles temas importan más.

**El truncamiento (láminas 141 y 142).** La SVD completa tiene tantos "temas" como documentos o palabras. **SVD truncado** se queda solo con los **r** más importantes y descarta el resto.

**Qué vamos a hacer:** aplicar SVD a la matriz TF-IDF de las 4 reseñas del Paso 1.0 y quedarnos con r = 2.

In [ ]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

U, S, Vt = np.linalg.svd(TFIDF_final.values, full_matrices=False)
print("Σ (importancia de cada tema):", S.round(3))

r = 2
temas_palabras = pd.DataFrame(Vt[:r], columns=TFIDF_final.columns, index=["tema 0", "tema 1"])
docs_temas = pd.DataFrame(U[:, :r] * S[:r], index=nombres_docs, columns=["tema 0", "tema 1"])
print("\nVᵀ: peso de cada palabra en cada tema")
display(temas_palabras.round(3))
print("U × Σ: cada documento resumido en 2 temas")
display(docs_temas.round(3))

**Qué deberías ver:**

- **Σ:** los dos primeros valores (1,306) son mucho mayores que los descartados (0,542). Los dos temas que conservamos explican casi todo.
- **Vᵀ:** un tema pone peso en batería (0,769), carga (0,527) y dura (0,361); el otro, en pantalla (0,769), brillo (0,527) y color (0,361). Nadie le dijo cuáles eran los temas.
- **U × Σ:** cada documento quedó resumido en 2 números en vez de 6, con 0,924 en su tema y 0 en el otro. Es **reducción de dimensionalidad**, como PCA.

Los signos pueden salir negativos: como en PCA, el signo es arbitrario. Lo que importa es qué palabras van juntas.

`TruncatedSVD` de scikit-learn hace lo mismo: `components_` es Vᵀ y `transform` es U × Σ.

In [ ]:
svd_j = TruncatedSVD(n_components=2, random_state=42).fit(TFIDF_final.values)
pd.DataFrame(np.abs(svd_j.components_), columns=TFIDF_final.columns).round(3)

**Tres cosas que confunden:**

1. **Con datos reales, el primer tema reúne lo más común de todo el corpus** (lo verán en el Paso 3.2). Los siguientes temas separan lo específico.
2. **Signos negativos:** son normales en LSA y no significan "en contra".
3. **LSA no da probabilidades.** Un documento puede tener 0,9 en un tema y -0,2 en otro: no son porcentajes. Esa es la ventaja de LDA.

## Paso 3.2. LSA con las reseñas

In [ ]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

N_TEMAS = 4
terminos = vect.get_feature_names_out()

lsa = TruncatedSVD(n_components=N_TEMAS, random_state=42).fit(X_tfidf)
for i, comp in enumerate(lsa.components_):
    print(f"LSA tema {i}: {', '.join(terminos[np.argsort(comp)[::-1][:6]])}")

**Qué deberías ver:** 4 listas de palabras. Como anticipamos, el primer tema de LSA casi siempre reúne las palabras más comunes de todo el corpus (resume "de qué habla el corpus en general"); los siguientes separan temas más específicos. Es el mismo `TruncatedSVD` que usamos para graficar en 2D.

## Paso 3.3. LDA paso a paso (láminas 144 a 154)

LDA (asignación latente de Dirichlet) también busca temas, pero con un **modelo de probabilidad** de cómo se escriben los documentos:

- **Cada documento es una mezcla de temas**, por ejemplo 70% batería y 30% pantalla. Se representa con **θ** (theta).
- **Cada tema es una mezcla de palabras**: el tema batería pone mucha probabilidad en "batería", "carga" y "dura". Se representa con **φ** (phi).

### El proceso generativo: razonamiento deductivo (láminas 145 y 152)

LDA imagina que cada documento se escribió así:

1. **Sortear la mezcla de temas del documento (θ).** Se sortea de una distribución de **Dirichlet**, que es una "distribución de distribuciones": genera listas de probabilidades que suman 1.
2. **Para cada palabra del documento:**
   - a. Sortear un **tema** según θ.
   - b. Sortear una **palabra** según la mezcla de palabras de ese tema (φ).

Si conociéramos θ y φ, podríamos **generar** documentos. Eso es el razonamiento deductivo: del modelo a los datos.

**Qué vamos a hacer:** generar un documento con un modelo inventado, para ver el proceso funcionando.

In [ ]:
rng = np.random.default_rng(0)
temas = ["batería", "pantalla"]
phi_inventado = {                     # mezcla de palabras de cada tema (cada fila suma 1)
    "batería":  {"batería": 0.5, "carga": 0.3, "dura": 0.2},
    "pantalla": {"pantalla": 0.5, "brillo": 0.3, "color": 0.2},
}
theta_doc = [0.7, 0.3]                # este documento es 70% batería y 30% pantalla

documento = []
for _ in range(8):
    tema = rng.choice(temas, p=theta_doc)                                             # paso 2a: sortear un tema
    palabra = rng.choice(list(phi_inventado[tema]), p=list(phi_inventado[tema].values()))  # paso 2b: sortear una palabra
    documento.append(f"{palabra} ({tema})")
print("Documento generado:", ", ".join(documento))

**Qué deberías ver:** 8 palabras, la mayoría del tema batería, y entre paréntesis el tema que se sorteó para cada una.

### La inferencia: razonamiento inductivo (lámina 153)

En la realidad es al revés: **tenemos los documentos y no conocemos θ ni φ.** Hay que deducir qué mezclas generaron más probablemente esos textos. Un método clásico es el **muestreo de Gibbs**, que se puede explicar en 4 pasos:

1. **Asignar un tema al azar a cada palabra** de cada documento.
2. **Tomar una palabra** y sacarla de los conteos.
3. **Calcular, para cada tema, qué tan probable es** que esa palabra sea de ese tema. Es el producto de dos cosas:
   - **¿Cuánto usa este documento ese tema?** (cuántas palabras del documento tienen ese tema)
   - **¿Cuánto usa ese tema esta palabra?** (en todo el corpus, cuántas veces esta palabra tiene ese tema)
4. **Sortear un tema nuevo** con esas probabilidades, volver a poner la palabra y pasar a la siguiente. Repetir muchas veces.

Con las vueltas, las palabras que aparecen juntas terminan en el mismo tema, y cada documento se concentra en pocos temas. Los parámetros **α** (alpha) y **β** (beta) son pequeños valores que se suman a los conteos para que ninguna probabilidad sea cero (como el suavizado de Laplace).

**Qué vamos a hacer:** programar estos 4 pasos y aplicarlos a las 4 reseñas del Paso 1.0. Lean el código con los comentarios: cada número corresponde a un paso.

In [ ]:
def lda_gibbs(docs, K, alpha=0.1, beta=0.1, iteraciones=200, semilla=1):
    rng = np.random.default_rng(semilla)
    vocab = sorted({w for d in docs for w in d}); idx = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
    # 1. tema al azar para cada palabra, y los conteos que lo resumen
    z = [[int(rng.integers(K)) for _ in d] for d in docs]
    n_dk = np.zeros((len(docs), K))     # palabras del documento d asignadas al tema k
    n_kw = np.zeros((K, V))             # veces que la palabra w está asignada al tema k
    n_k = np.zeros(K)                   # total de palabras asignadas al tema k
    for d, doc in enumerate(docs):
        for i, w in enumerate(doc):
            k = z[d][i]; n_dk[d, k] += 1; n_kw[k, idx[w]] += 1; n_k[k] += 1
    inicial = [list(x) for x in z]
    for _ in range(iteraciones):
        for d, doc in enumerate(docs):
            for i, w in enumerate(doc):
                k = z[d][i]; n_dk[d, k] -= 1; n_kw[k, idx[w]] -= 1; n_k[k] -= 1        # 2. sacar la palabra
                p = (n_dk[d] + alpha) * (n_kw[:, idx[w]] + beta) / (n_k + V * beta)    # 3. documento usa tema × tema usa palabra
                k = int(rng.choice(K, p=p / p.sum()))                                  # 4. sortear tema nuevo
                z[d][i] = k; n_dk[d, k] += 1; n_kw[k, idx[w]] += 1; n_k[k] += 1        #    y volver a ponerla
    theta = (n_dk + alpha) / (n_dk + alpha).sum(axis=1, keepdims=True)
    phi = (n_kw + beta) / (n_kw + beta).sum(axis=1, keepdims=True)
    return vocab, theta, phi, inicial, z

docs_tokens = [d.split() for d in docs_juguete]
vocab_j, theta, phi, z_inicial, z_final = lda_gibbs(docs_tokens, K=2)

print("Temas al azar al inicio:", z_inicial)
print("Temas al final:         ", z_final)
print("\nφ: mezcla de palabras de cada tema")
display(pd.DataFrame(phi, columns=vocab_j, index=["tema 0", "tema 1"]).round(2))
print("θ: mezcla de temas de cada documento")
display(pd.DataFrame(theta, index=nombres_docs, columns=["tema 0", "tema 1"]).round(2))

**Qué deberías ver:**

- **Al inicio,** los temas están mezclados al azar dentro de cada documento.
- **Al final,** las palabras de los docs 1 y 2 tienen un tema y las de los docs 3 y 4, el otro.
- **φ:** un tema con batería (0,47), carga (0,32) y dura (0,17); el otro con pantalla, brillo y color.
- **θ:** cada documento es 97% de su tema. A diferencia de LSA, **son probabilidades que suman 1**.

`LatentDirichletAllocation` de scikit-learn resuelve lo mismo con otro método de inferencia (bayesiano variacional), más rápido para corpus grandes. La lógica es la misma: deducir θ y φ desde los conteos.

### LSA contra LDA

| | LSA | LDA |
|---|---|---|
| Entrada | Matriz TF-IDF | Matriz de **conteos** |
| Cómo encuentra los temas | Álgebra lineal (SVD) | Modelo de probabilidad (inferencia) |
| Pesos de los documentos | Números con signo, no son porcentajes | Probabilidades que suman 1 |
| Resultado igual cada vez | Sí | Puede variar con la semilla |
| Cuándo conviene | Rápido, buen punto de partida, para reducir dimensiones | Cuando se quiere interpretar "este documento es 70% de este tema" |

## Paso 3.4. LDA con las reseñas

LDA trabaja con **conteos** (como Naive Bayes), no con TF-IDF.

In [ ]:
vect_conteo = CountVectorizer(min_df=2)
X_conteo = vect_conteo.fit_transform(resenas["para_temas"])
terminos_c = vect_conteo.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=N_TEMAS, random_state=42).fit(X_conteo)
for i, comp in enumerate(lda.components_):
    print(f"LDA tema {i}: {', '.join(terminos_c[np.argsort(comp)[::-1][:6]])}")

mezcla = lda.transform(X_conteo)                        # cada fila: probabilidad de cada tema en esa reseña
print("\nPrimera reseña:", resenas.loc[0, "completo"][:90])
print("Mezcla de temas:", mezcla[0].round(2), "(suma:", round(mezcla[0].sum(), 2), ")")

**Qué deberías ver:** 4 temas con sus palabras y, para la primera reseña, 4 probabilidades que suman 1. Esa es la diferencia clave con K-medias: **una reseña puede hablar 60% de batería y 40% de pantalla**.

**Ojo:** con pocos documentos, LDA da temas inestables (cambian con `random_state`). Funciona bien desde algunos miles de documentos.

### Ejercicio 4
Guarda en `tema_principal_0` el número del tema de LDA con **mayor probabilidad** en la primera reseña (`mezcla[0]`).

In [ ]:
tema_principal_0 = None  # TU CÓDIGO AQUÍ
print(tema_principal_0)

In [ ]:
revisar(lambda: tema_principal_0 == int(np.argmax(mezcla[0])))

---
# Parte 4. Skip-gram: vectores de palabras (láminas 159 a 167)

El modelo de n-gramas de la Clase 12 existe para **predecir**. Skip-gram (Word2Vec) también predice, pero solo como excusa: lo que queremos son **vectores** (embeddings) donde palabras de significado parecido queden cerca. Resuelve el problema del one-hot y de TF-IDF: "celular" y "teléfono" deberían quedar cerca aunque sean palabras distintas.

## Paso 4.0. ¿Qué es un embedding? (láminas 156 a 158)

**El problema del one-hot (Clase 12):** cada palabra es un vector enorme con un solo 1. Todas las palabras están a la misma distancia entre sí: "batería" está igual de lejos de "pila" que de "pantalla". El vector no dice nada del significado.

> **Definición. Embedding (incrustación de palabras):** un vector **corto y denso** (por ejemplo, 50 números, casi todos distintos de cero) que representa una palabra, de modo que **palabras de significado parecido tengan vectores parecidos**.

| | One-hot | Embedding |
|---|---|---|
| Largo | Tamaño del vocabulario (miles) | Elegido por nosotros (16, 50, 300) |
| Valores | Un 1 y el resto ceros | Números reales, casi todos distintos de cero |
| Significado | Ninguno | Palabras parecidas, vectores cercanos |
| Cómo se obtiene | Se construye directamente | **Se aprende** de los datos |

El ejemplo clásico de la lámina 158: con buenos embeddings, **rey − hombre + mujer ≈ reina**. Las diferencias entre vectores capturan relaciones.

### Un embedding es una tabla

Una **capa de embedding** es simplemente una tabla con una fila por palabra del vocabulario y tantas columnas como dimensiones. Para obtener el vector de una palabra, se busca su fila usando su **número** de la codificación entera (Clase 12). Multiplicar el one-hot por la tabla da exactamente lo mismo: el 1 "selecciona" la fila.

In [ ]:
rng = np.random.default_rng(0)
vocab_mini = ["<PAD>", "<OOV>", "batería", "pantalla", "carga"]
tabla = rng.normal(size=(len(vocab_mini), 3)).round(2)       # 5 palabras × 3 dimensiones (al inicio, al azar)
display(pd.DataFrame(tabla, index=vocab_mini, columns=["dim 0", "dim 1", "dim 2"]))

indice = vocab_mini.index("batería")                         # codificación entera: batería = 2
one_hot_bat = np.eye(len(vocab_mini))[indice]                # one-hot de batería
print("Fila 2 de la tabla:     ", tabla[indice])
print("One-hot × tabla:        ", one_hot_bat @ tabla)

**Qué deberías ver:** la misma fila dos veces. Por eso las redes reciben la **codificación entera** (no el one-hot): buscar la fila es mucho más eficiente.

Al inicio la tabla es aleatoria. **Entrenar** los embeddings significa ajustar esos números para que cumplan una tarea. Word2Vec lo hace con una tarea de predicción de palabras, que vemos ahora.

### Cómo aprende Word2Vec (láminas 159 a 167)

Word2Vec es una red neuronal muy pequeña:

1. **Entrada:** una palabra, como su número (o su one-hot).
2. **Capa oculta:** la tabla de embeddings. La palabra pasa a ser su vector.
3. **Salida:** una probabilidad para cada palabra del vocabulario.

Hay dos formas de plantear la tarea:

- **CBOW:** con las palabras de alrededor, predecir la palabra central. "como una ___ todas las" → "manzana".
- **Skip-gram:** con la palabra central, predecir las palabras de alrededor. "manzana" → "como", "una", "todas", "las".

Durante el entrenamiento, la red se equivoca, se calcula el error y se ajustan los números de la tabla (como en cualquier red neuronal del Capítulo 8). **Al final, la predicción se descarta y nos quedamos con la tabla.** Palabras que aparecen en contextos parecidos terminan con vectores parecidos, porque la red necesita que produzcan predicciones parecidas.

## Paso 4.1. Los pares de entrenamiento

Skip-gram toma cada palabra como **central** y trata de predecir las palabras de alrededor, dentro de una **ventana**. Con la frase de la lámina 157 y una ventana de 2:

In [ ]:
frase = "como una manzana todas las mañanas".split()
VENTANA = 2

pares = []
for i, central in enumerate(frase):
    for j in range(max(0, i - VENTANA), min(len(frase), i + VENTANA + 1)):
        if j != i:
            pares.append((central, frase[j]))

print(f"{len(pares)} pares (central, contexto):")
for p in pares:
    print(" ", p)

**Qué deberías ver:** pares como `('manzana', 'como')`, `('manzana', 'una')`, `('manzana', 'todas')` y `('manzana', 'las')`. La red aprende a predecir el segundo elemento a partir del primero, y al hacerlo ajusta un vector para cada palabra.

**CBOW** (la otra variante) hace lo contrario: con las palabras de alrededor, predice la central.

### Ejercicio 5
¿Cuántos pares se generan con la misma frase y una **ventana de 1**? Guárdalo en `n_pares_v1`.

In [ ]:
n_pares_v1 = None  # TU CÓDIGO AQUÍ
print(n_pares_v1)

In [ ]:
revisar(lambda: n_pares_v1 == 10)

## Paso 4.2. Entrenar Word2Vec con las reseñas

**Qué vamos a hacer:** entrenar skip-gram (`sg=1`) con `gensim` sobre las oraciones de las reseñas, y preguntar qué palabras quedan más cerca de algunas palabras clave.

In [ ]:
from gensim.models import Word2Vec

oraciones = [tokens_simples(o) for texto in resenas["completo"] for o in re.split(r"[.!?]", texto)]
oraciones = [o for o in oraciones if len(o) > 2]

w2v = Word2Vec(oraciones, vector_size=50, window=3, min_count=2, sg=1, epochs=200, seed=42, workers=1)
print("Palabras con vector:", len(w2v.wv))
print("Vector de 'batería' (primeros 8 de 50 valores):", w2v.wv["batería"][:8].round(2))

for palabra in ["batería", "pantalla", "celular"]:
    if palabra in w2v.wv:
        print(f"\nMás parecidas a '{palabra}':", [(p, round(s, 2)) for p, s in w2v.wv.most_similar(palabra, topn=5)])

**Qué deberías ver:** cada palabra tiene ahora un vector de 50 números, y una lista de palabras "parecidas" según la similitud coseno entre vectores.

**Seamos honestos con el resultado:** con unos cientos de oraciones, algunas palabras parecidas tendrán sentido (palabras que aparecen en las mismas frases) y otras no. Word2Vec necesita **millones** de palabras para que los vectores capturen bien el significado. Por eso, en la práctica, se usan vectores **ya entrenados** con grandes corpus, o modelos como los del Capítulo 10, que ya traen esa representación.

---
# Parte 5. Clasificar reseñas con una red LSTM (láminas 168 a 175)

Esta parte adelanta el Capítulo 8 (redes neuronales con Keras). Juntamos todo lo del capítulo: **codificación entera + relleno + embedding + una red que lee en orden**.

## Paso 5.1. ¿Por qué importa el orden?

TF-IDF y la bolsa de palabras **pierden el orden**. Para ellas, "no es malo, es bueno" y "no es bueno, es malo" son el mismo documento: tienen exactamente las mismas palabras. Una persona lee de izquierda a derecha y va acumulando el sentido. Eso es lo que hace una **red recurrente**.

## Paso 5.2. Red recurrente (RNN)

Una RNN lee la secuencia **palabra por palabra** y mantiene una **memoria**: un vector llamado **estado oculto** $h$. En cada paso combina la palabra actual con lo que recordaba:

$$h_t = \\tanh(W \\cdot x_t + U \\cdot h_{t-1} + b)$$

- $x_t$: el embedding de la palabra número $t$.
- $h_{t-1}$: la memoria después de leer la palabra anterior.
- $W$, $U$, $b$: los números que la red aprende.

Al terminar la última palabra, $h$ resume toda la reseña, y una capa final lo convierte en "positiva" o "negativa".

**El problema:** en secuencias largas, la memoria se "diluye": lo que se leyó al principio casi no influye al final (en el entrenamiento, los gradientes se desvanecen). Si la reseña empieza con "no" y la palabra clave viene 20 palabras después, la RNN simple ya lo olvidó.

## Paso 5.3. LSTM: una memoria con puertas

LSTM (*Long Short-Term Memory*) agrega una segunda memoria, el **estado de la celda** $c$, que funciona como una **cinta transportadora**: la información puede viajar por ella sin diluirse. Tres **puertas** deciden qué pasa con esa cinta en cada palabra. Cada puerta es un número entre 0 y 1 (una sigmoide): 0 es "cerrado" y 1 es "abierto".

| Puerta | Pregunta que responde | Fórmula |
|---|---|---|
| **Olvido** $f_t$ | ¿Qué parte de la memoria anterior borro? | $f_t = \\sigma(W_f [h_{t-1}, x_t] + b_f)$ |
| **Entrada** $i_t$ | ¿Qué parte de la información nueva guardo? | $i_t = \\sigma(W_i [h_{t-1}, x_t] + b_i)$ |
| **Salida** $o_t$ | ¿Qué parte de la memoria muestro en este paso? | $o_t = \\sigma(W_o [h_{t-1}, x_t] + b_o)$ |

Con las puertas, la memoria se actualiza así:

1. **Candidato:** $\\tilde{c}_t = \\tanh(W_c [h_{t-1}, x_t] + b_c)$, la información nueva que podría guardarse.
2. **Nueva celda:** $c_t = f_t \\cdot c_{t-1} + i_t \\cdot \\tilde{c}_t$, lo que queda de lo anterior más lo nuevo que se admite.
3. **Nuevo estado oculto:** $h_t = o_t \\cdot \\tanh(c_t)$, lo que se muestra hacia afuera.

**Con un ejemplo:** "**no** me gustó la pantalla, aunque la batería es buena".

- Al leer "**no**", la puerta de **entrada** se abre: es información importante y se guarda en la cinta.
- En "me", "la" y "aunque", la puerta de **olvido** queda cerca de 1: no se borra el "no".
- Al leer "gustó", la red combina "no" con "gustó" y la celda registra algo negativo.

Nadie programa estas reglas: la red aprende cuándo abrir y cerrar cada puerta a partir de los ejemplos.

## Paso 5.4. La arquitectura (lámina 168: "entrada de secuencia y salida vectorial")

| Capa | Recibe | Entrega | Qué hace |
|---|---|---|---|
| `Embedding` | Secuencia de 25 números enteros | 25 vectores de 16 números | Busca el embedding de cada palabra en la tabla (Paso 4.0) |
| `LSTM(16)` | Los 25 vectores, en orden | 1 vector de 16 números | Lee la reseña y entrega su memoria final $h$ |
| `Dense(1, sigmoid)` | El vector de 16 | 1 número entre 0 y 1 | Probabilidad de que la reseña sea positiva |

`mask_zero=True` en el embedding le dice a la LSTM que **ignore los ceros del relleno**. Si la LSTM debiera entregar un vector **por cada palabra** (para apilar otra LSTM encima), se usa `return_sequences=True` (lámina 169).

## Paso 5.5. Preparar los datos

**Qué vamos a hacer:** usar las reseñas positivas y negativas (sin las de nota 3), el texto **para opinión** (que conserva "no"), un vocabulario completo y un relleno a 25 palabras. Separamos 25% para prueba.

In [ ]:
import keras
from keras import layers
from sklearn.model_selection import train_test_split

op = resenas[resenas["nota"] != 3].copy()
y_op = (op["nota"] >= 4).astype(int).values                   # 1 = positiva, 0 = negativa
textos_op = op["para_opinion"].tolist()

frec_op = Counter(t for x in textos_op for t in x.split())
vocab_op = {p: i for i, (p, _) in enumerate(frec_op.most_common(), start=2)}   # 0 relleno, 1 desconocida
seqs_op = [[vocab_op.get(t, 1) for t in x.split()] for x in textos_op]

LARGO_LSTM = 25
X_op = rellenar(seqs_op, LARGO_LSTM)
X_tr, X_te, y_tr, y_te = train_test_split(X_op, y_op, test_size=0.25, random_state=42, stratify=y_op)
pesos = {0: len(y_tr) / (2 * (y_tr == 0).sum()), 1: len(y_tr) / (2 * (y_tr == 1).sum())}   # como class_weight="balanced"

print("Vocabulario:", len(vocab_op), "| Entrenamiento:", X_tr.shape, "| Prueba:", X_te.shape)
print("Pesos por clase:", {k: round(float(v), 2) for k, v in pesos.items()})

**Qué deberías ver:** el tamaño del vocabulario, las formas de los conjuntos (25 columnas) y un peso mayor para la clase negativa, que es la minoritaria.

## Paso 5.6. Construir el modelo

In [ ]:
keras.utils.set_random_seed(42)
modelo_lstm = keras.Sequential([
    layers.Input(shape=(LARGO_LSTM,)),
    layers.Embedding(input_dim=len(vocab_op) + 2, output_dim=16, mask_zero=True),
    layers.LSTM(16),
    layers.Dense(1, activation="sigmoid"),
])
modelo_lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo_lstm.summary()

**Qué deberías ver:** una tabla con las 3 capas y sus parámetros. Cómo se calculan:

- **Embedding:** (tamaño del vocabulario + 2) × 16. Es la tabla del Paso 4.0: una fila de 16 números por palabra.
- **LSTM:** 4 × (16 × (16 + 16) + 16) = **2.112**. Hay 4 bloques (3 puertas y el candidato), y cada uno combina la memoria anterior (16) con la palabra actual (16), más un sesgo.
- **Dense:** 16 pesos + 1 sesgo = **17**.

### Ejercicio 6
Guarda en `params_embedding` el número de parámetros de la capa Embedding, **calculado con la fórmula** (no copiado de la tabla).

In [ ]:
params_embedding = None  # TU CÓDIGO AQUÍ
print(params_embedding)

In [ ]:
revisar(lambda: params_embedding == modelo_lstm.layers[0].count_params())

## Paso 5.7. Entrenar y evaluar

- **`epochs=40`:** la red recorre los datos de entrenamiento 40 veces.
- **`batch_size=8`:** ajusta los pesos cada 8 reseñas.
- **`class_weight`:** los errores en la clase negativa pesan más.

In [ ]:
from sklearn.metrics import classification_report

historia = modelo_lstm.fit(X_tr, y_tr, epochs=40, batch_size=8, class_weight=pesos, verbose=0)
print("Exactitud en entrenamiento (última época):", round(historia.history["accuracy"][-1], 3))

prob_te = modelo_lstm.predict(X_te, verbose=0).ravel()
print(classification_report(y_te, (prob_te >= 0.5).astype(int), target_names=["negativa", "positiva"], digits=3))

plt.plot(historia.history["loss"]); plt.title("Pérdida en entrenamiento por época"); plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.show()

**Qué deberías ver:** la pérdida baja época a época y la exactitud en entrenamiento llega a 1. Pero en prueba el resultado es bastante peor: con el respaldo, un F1 negativo cercano a 0,5, peor que la regresión logística con TF-IDF de la Parte 2 (0,75).

**Esto es sobreajuste, y es la lección más importante de la parte.** Una LSTM tiene miles de parámetros y aprende de memoria cuando hay pocos ejemplos. Con unas decenas de reseñas, el modelo simple gana. Las LSTM y los embeddings brillan con **miles o millones** de textos, o partiendo de embeddings ya entrenados. Elegir el modelo según la cantidad de datos es parte del trabajo.

**Otras arquitecturas de las láminas 170 y 171:** una capa `Conv1D` antes de la LSTM busca patrones locales de 2 o 3 palabras seguidas (como "no me gustó") y luego la LSTM los lee en orden. Las veremos con más calma en los Capítulos 8 y 9.

---
# Cierre

| Parte | Lo esencial |
|---|---|
| Codificación entera | Cada palabra a un número; las más frecuentes, los números más bajos. 0 para relleno y 1 para desconocidas. |
| Relleno | Todas las secuencias del mismo largo, el formato que necesitan las redes neuronales. |
| Similitud coseno | Compara la dirección de dos vectores; no le importa el largo del documento. |
| Naive Bayes | Probabilidad de la clase por las probabilidades de cada palabra en esa clase. |
| LSA y LDA | Temas como componentes (LSA) o como mezclas de probabilidad (LDA). |
| N-gramas | P(palabra \| anterior) = conteo del par / conteo de la anterior. Se predice la de mayor probabilidad. |
| Laplace | Sumar 1 para que nada tenga probabilidad cero. |
| Embeddings | Vectores cortos y aprendidos: palabras parecidas, vectores cercanos. Una capa de embedding es una tabla. |
| Skip-gram | Predecir el contexto para aprender vectores de palabras. |
| LSTM | Lee en orden y guarda memoria con puertas de olvido, entrada y salida. Necesita muchos datos. |

Guarden el notebook en `ejercicios_estudiantes` como `clase13_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
"color" aparece en 1 de 4 documentos: ln(5/2) + 1 = 1,916.

In [ ]:
idf_color = np.log((1 + 4) / (1 + 1)) + 1
print(round(idf_color, 3))

### Ejercicio 2
B · D = 1·2 + 1·2 + 0·1 = 4. |B| = √2. |D| = √9 = 3. cos = 4 / (√2 · 3) = 0,943.

In [ ]:
cos_bd = B @ D / (np.linalg.norm(B) * np.linalg.norm(D))
print(round(cos_bd, 3))

### Ejercicio 3

In [ ]:
_f1 = {n: f1_score(opiniones["sentimiento"], p, pos_label="negativa") for n, p in predicciones.items()}
mejor_f1_negativa = max(_f1, key=_f1.get)
print(mejor_f1_negativa, {n: round(v, 3) for n, v in _f1.items()})

### Ejercicio 4

In [ ]:
tema_principal_0 = int(np.argmax(mezcla[0]))
print(tema_principal_0)

### Ejercicio 5
Con ventana 1, cada palabra se empareja solo con sus vecinas inmediatas. Las 2 palabras de los extremos tienen 1 vecina y las 4 del medio tienen 2: 2·1 + 4·2 = 10.

In [ ]:
n_pares_v1 = sum(1 for i in range(len(frase)) for j in range(max(0, i - 1), min(len(frase), i + 2)) if j != i)
print(n_pares_v1)

### Ejercicio 6
Una fila de 16 números por cada palabra del vocabulario, más las 2 filas reservadas para relleno y desconocidas.

In [ ]:
params_embedding = (len(vocab_op) + 2) * 16
print(params_embedding)